# LeetCode #161 — One Edit Distance: đúng một phép sửa

**Mức độ:** Medium · **Chủ đề:** Two Pointers, String

Skeleton trên LeetCode:

```python
class Solution:
    def isOneEditDistance(self, s: str, t: str) -> bool:
```

Mục tiêu của notebook:

1. Ba phép sửa: **chèn**, **xóa**, **thay** một ký tự — và quan hệ với độ dài hai chuỗi.
2. Baseline: tính **edit distance** (#72) bằng DP `O(nm)` rồi so với 1.
3. Lời giải `O(n)`: tìm **vị trí lệch đầu tiên**, rồi so phần còn lại theo một trong hai trường hợp.
4. Chú ý: **đúng một** phép sửa — hai chuỗi bằng nhau thì trả `False`.

# 1. Bài toán là gì?

Cho hai chuỗi `s` và `t`, trả về `True` nếu chúng cách nhau **đúng một** phép sửa, ngược lại `False`.

`s` được gọi là cách `t` một phép sửa nếu có thể:

- **chèn** đúng một ký tự vào `s` để được `t`, hoặc
- **xóa** đúng một ký tự khỏi `s` để được `t`, hoặc
- **thay** đúng một ký tự của `s` bằng một ký tự **khác** để được `t`.

**Constraints (theo đề):**

- `0 <= s.length, t.length <= 10^4`
- `s` và `t` gồm chữ cái thường, chữ cái hoa và chữ số.

---

## Ví dụ

```text
s = "ab",  t = "acb"   ->  True    (chèn 'c')
s = "",    t = ""      ->  False   (cần 0 phép sửa)
s = "a",   t = ""      ->  True    (xóa 'a')
s = "abc", t = "abd"   ->  True    (thay 'c' → 'd')
s = "abc", t = "bcd"   ->  False
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Chuỗi (string) trong Python
2. Hai con trỏ (two pointers)
3. DP hai chiều (bảng `dp[i][j]`)

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Chuỗi (string) trong Python

### Chuỗi là bất biến

`str` **không sửa được** tại chỗ: `s[0] = "x"` gây lỗi. Mọi "sửa" đều tạo chuỗi mới. Vì vậy:

- Ghép nhiều mảnh: gom vào **list** rồi `"".join(list)` (`O(n)`), tránh `s += ...` trong vòng lặp (có thể `O(n²)`).
- Cần sửa từng ký tự: `chars = list(s)`, sửa, rồi `"".join(chars)`.

In [1]:
s_str = "hello"
try:
    s_str[0] = "H"
except TypeError as e:
    print("Sửa trực tiếp:", e)
chars_str = list(s_str); chars_str[0] = "H"
print("qua list:", "".join(chars_str))
manh = []
for i in range(5):
    manh.append(str(i))
print("join:", "-".join(manh))

Sửa trực tiếp: 'str' object does not support item assignment
qua list: Hello
join: 0-1-2-3-4


### Cắt chuỗi (slicing) và các thao tác hay dùng

| Biểu thức | Ý nghĩa |
|---|---|
| `s[i:j]` | ký tự từ `i` tới `j−1` (tạo chuỗi mới, `O(j−i)`) |
| `s[::-1]` | chuỗi đảo ngược |
| `s.split()` | tách theo khoảng trắng (bỏ khoảng trắng thừa) |
| `s.split(",")` | tách theo dấu phẩy (giữ phần rỗng) |
| `s.strip()` | bỏ khoảng trắng hai đầu |
| `s.startswith(p)`, `s.find(p)` | kiểm tra tiền tố, tìm vị trí |
| `c.isdigit()`, `c.isalpha()`, `c.isalnum()`, `c.isspace()` | phân loại ký tự |
| `c.lower()`, `c.upper()` | đổi hoa/thường |

In [2]:
t_str = "  Xin chào   thế giới  "
print(repr(t_str.split()), "|", repr(t_str.strip()), "|", repr("a,,b".split(",")))
print("đảo:", "abcdef"[::-1], "| cắt [1:4]:", "abcdef"[1:4], "| 'abc'.find('c') =", "abc".find("c"))
print([(c, c.isdigit(), c.isalpha()) for c in "a1 "])

['Xin', 'chào', 'thế', 'giới'] | 'Xin chào   thế giới' | ['a', '', 'b']
đảo: fedcba | cắt [1:4]: bcd | 'abc'.find('c') = 2
[('a', False, True), ('1', True, False), (' ', False, False)]


### Ký tự và mã số: `ord` / `chr`

Mỗi ký tự có một mã số (Unicode). `ord("a") = 97`, `chr(97) = "a"`. Mẹo: chữ số `c` → giá trị `ord(c) - ord("0")`; chữ thường `c` → chỉ số 0..25 là `ord(c) - ord("a")`.

In [3]:
print("ord('a') =", ord("a"), "| chr(98) =", chr(98))
print("'7' → 7:", ord("7") - ord("0"), "| 'd' → chỉ số:", ord("d") - ord("a"))
dem26 = [0] * 26
for c in "banana":
    dem26[ord(c) - ord("a")] += 1
print("đếm 26 chữ cái của 'banana' (a, b, n):", dem26[0], dem26[1], dem26[13])

ord('a') = 97 | chr(98) = b
'7' → 7: 7 | 'd' → chỉ số: 3
đếm 26 chữ cái của 'banana' (a, b, n): 3 1 2


### Gom nhóm ký tự liên tiếp

`itertools.groupby` gom các phần tử **liên tiếp bằng nhau** — tiện cho mã hóa độ dài loạt (run-length): `"aaabcc" → 3a 1b 2c`. Tự viết bằng hai con trỏ: `i` ở đầu nhóm, `j` chạy tới khi ký tự khác.

In [4]:
from itertools import groupby
print([(k, len(list(g))) for k, g in groupby("aaabcc")])

def run_length(s):
    out, i = [], 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        out.append(f"{j - i}{s[i]}")
        i = j
    return "".join(out)
print("run_length('aaabcc') =", run_length("aaabcc"))

[('a', 3), ('b', 1), ('c', 2)]
run_length('aaabcc') = 3a1b2c


Lỗi hay gặp: `split(" ")` sinh phần rỗng khi có nhiều dấu cách; so sánh chuỗi số `"10" < "9"` là **True** (so từ điển, không phải số); nối chuỗi trong vòng lặp lớn; quên chuỗi rỗng.

## 2.2 Hai con trỏ (two pointers)

### Ý tưởng

Dùng **hai chỉ số** di chuyển trên mảng/chuỗi theo quy tắc, để thay vòng lặp lồng `O(n²)` bằng một lượt `O(n)`. Hai kiểu chính:

**1. Hai đầu tiến vào nhau** (thường trên mảng đã sắp xếp):

```text
[1, 2, 4, 7, 11, 15], tìm hai số có tổng 15
 L                 R     1 + 15 = 16 > 15 → R lùi (cần nhỏ hơn)
 L             R         1 + 11 = 12 < 15 → L tiến (cần lớn hơn)
    L          R         2 + 11 = 13 < 15 → L tiến
       L       R         4 + 11 = 15 ✓
```

Vì sao được phép bỏ? Khi `a[L] + a[R] > target`, mọi cặp `(L', R)` với `L' > L` còn lớn hơn → `R` không thể là đáp án, bỏ `R`.

**2. Cùng chiều — con trỏ đọc và con trỏ ghi** (sửa mảng tại chỗ):

```text
xóa mọi số 0, giữ thứ tự: [0, 1, 0, 3, 12]
read đi qua từng phần tử; gặp số khác 0 thì chép vào vị trí write, write += 1
→ [1, 3, 12, ...]
```

In [5]:
def tong_hai_so_da_sap_xep(a, target):
    L, R = 0, len(a) - 1
    while L < R:
        s = a[L] + a[R]
        print(f"   L={L} R={R}: {a[L]} + {a[R]} = {s}")
        if s == target:
            return L, R
        if s < target:
            L += 1
        else:
            R -= 1
    return None

print("→", tong_hai_so_da_sap_xep([1, 2, 4, 7, 11, 15], 15))

def bo_so_khong(a):
    write = 0
    for read in range(len(a)):
        if a[read] != 0:
            a[write] = a[read]
            write += 1
    return a[:write]

print("bỏ số 0:", bo_so_khong([0, 1, 0, 3, 12]))

def la_palindrome(s):
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            return False
        i += 1; j -= 1
    return True
print("'racecar' palindrome?", la_palindrome("racecar"), "| 'abca'?", la_palindrome("abca"))

   L=0 R=5: 1 + 15 = 16
   L=0 R=4: 1 + 11 = 12
   L=1 R=4: 2 + 11 = 13
   L=2 R=4: 4 + 11 = 15
→ (2, 4)
bỏ số 0: [1, 3, 12]
'racecar' palindrome? True | 'abca'? False


### Đảo ngược một đoạn tại chỗ

Hai con trỏ ở hai đầu, đổi chỗ rồi tiến vào trong — dùng rất nhiều (đảo mảng, xoay mảng, đảo từ).

In [6]:
def dao_doan(a, i, j):
    while i < j:
        a[i], a[j] = a[j], a[i]
        i += 1; j -= 1

arr_tp = [1, 2, 3, 4, 5, 6]
dao_doan(arr_tp, 1, 4)
print("đảo đoạn [1..4]:", arr_tp)

đảo đoạn [1..4]: [1, 5, 4, 3, 2, 6]


### Lỗi hay gặp

- Quên **sắp xếp** trước khi dùng hai đầu cho bài tổng.
- Điều kiện `L <= R` thay vì `L < R` → dùng một phần tử hai lần.
- Không chứng minh được vì sao bỏ một đầu là an toàn → dễ bỏ mất đáp án.
- Con trỏ ghi vượt con trỏ đọc (không thể xảy ra nếu `write` chỉ tăng khi ghi).

## 2.3 DP hai chiều (bảng `dp[i][j]`)

### Khi nào cần bảng hai chiều?

Khi trạng thái cần **hai** thông số, thường gặp ở:

- **Lưới**: `dp[i][j]` = đáp án cho ô `(i, j)` (số đường đi, tổng nhỏ nhất…).
- **Hai chuỗi**: `dp[i][j]` = đáp án cho **tiền tố** `s[:i]` và `t[:j]` (edit distance, LCS, xen kẽ…).
- **Đoạn con**: `dp[i][j]` = đáp án cho đoạn `s[i..j]` (palindrome…).

### Ví dụ 1: số đường đi trên lưới

Robot ở góc trên-trái, mỗi bước đi **phải** hoặc **xuống**. Có bao nhiêu đường tới ô `(i, j)`? Bước cuối đến từ ô **trên** hoặc ô **trái**:

```text
dp[i][j] = dp[i-1][j] + dp[i][j-1]
hàng 0 và cột 0 đều = 1 (chỉ một cách: đi thẳng)
```

In [7]:
def so_duong(m, n):
    dp = [[0] * n for _ in range(m)]      # KHÔNG dùng [[0] * n] * m
    for i in range(m):
        for j in range(n):
            if i == 0 or j == 0:
                dp[i][j] = 1
            else:
                dp[i][j] = dp[i - 1][j] + dp[i][j - 1]
    return dp

for row in so_duong(3, 5):
    print(" ".join(f"{v:>3}" for v in row))

  1   1   1   1   1
  1   2   3   4   5
  1   3   6  10  15


Thứ tự điền: theo hàng từ trên xuống, trong hàng từ trái sang → khi tính `dp[i][j]`, ô trên và ô trái **đã có**.

**Tiết kiệm bộ nhớ:** hàng `i` chỉ cần hàng `i−1` → giữ một mảng một chiều `row`, cập nhật `row[j] += row[j-1]` (giá trị cũ của `row[j]` chính là ô trên).

In [8]:
def so_duong_mot_hang(m, n):
    row = [1] * n
    for _ in range(1, m):
        for j in range(1, n):
            row[j] += row[j - 1]          # row[j] (cũ) = trên, row[j-1] (mới) = trái
    return row[-1]
print("so_duong_mot_hang(3, 7) =", so_duong_mot_hang(3, 7), "| so_duong(3, 7) =", so_duong(3, 7)[-1][-1])

so_duong_mot_hang(3, 7) = 28 | so_duong(3, 7) = 28


### Ví dụ 2: hai chuỗi — dãy con chung dài nhất (LCS)

`dp[i][j]` = độ dài dãy con chung dài nhất của `s[:i]` và `t[:j]`. Xét ký tự **cuối** của hai tiền tố:

- `s[i-1] == t[j-1]` → dùng nó: `dp[i][j] = dp[i-1][j-1] + 1`.
- khác → bỏ ký tự cuối của một trong hai: `dp[i][j] = max(dp[i-1][j], dp[i][j-1])`.

Hàng 0 và cột 0 ứng với **chuỗi rỗng** → bằng 0. Đó là lý do bảng có kích thước `(n+1) × (m+1)`.

In [9]:
def lcs(s, t):
    n, m = len(s), len(t)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s[i - 1] == t[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp

s_demo, t_demo = "abcde", "ace"
bang = lcs(s_demo, t_demo)
print("      ''  " + "   ".join(t_demo))
for i, row in enumerate(bang):
    print(f"{('' if i == 0 else s_demo[i - 1])!r:>4} " + " ".join(f"{v:>3}" for v in row))
print("LCS =", bang[-1][-1])

      ''  a   c   e
  ''   0   0   0   0
 'a'   0   1   1   1
 'b'   0   1   1   1
 'c'   0   1   2   2
 'd'   0   1   2   2
 'e'   0   1   2   3
LCS = 3


### Ví dụ 3: khoảng cách sửa (edit distance) — số phép chèn/xóa/thay ít nhất

`dp[i][j]` = số phép sửa ít nhất biến `s[:i]` thành `t[:j]`.

- Ký tự cuối bằng nhau → `dp[i-1][j-1]`.
- Khác → `1 + min(thay: dp[i-1][j-1], xóa s[i-1]: dp[i-1][j], chèn t[j-1]: dp[i][j-1])`.
- Cơ sở: `dp[i][0] = i` (xóa hết), `dp[0][j] = j` (chèn hết).

In [10]:
def khoang_cach_sua(s, t):
    n, m = len(s), len(t)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): dp[i][0] = i
    for j in range(m + 1): dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s[i - 1] == t[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j - 1], dp[i - 1][j], dp[i][j - 1])
    return dp[n][m]

for a_, b_ in [("horse", "ros"), ("kitten", "sitting"), ("", "abc")]:
    print(f"{a_!r} → {b_!r}: {khoang_cach_sua(a_, b_)}")

'horse' → 'ros': 3
'kitten' → 'sitting': 3
'' → 'abc': 3


### Lỗi hay gặp

- `[[0] * n] * m` tạo `m` hàng là **cùng một** list → sửa một ô thì cả cột đổi.
- Lệch chỉ số: bảng `(n+1)×(m+1)` nhưng ký tự là `s[i-1]`, `t[j-1]`.
- Quên hàng/cột cơ sở (chuỗi rỗng, biên lưới).
- Khi tối ưu một hàng: ghi đè giá trị cần dùng lại (ví dụ `dp[i-1][j-1]`) — phải lưu tạm trước.

In [11]:
sai = [[0] * 3] * 2
sai[0][0] = 9
dung = [[0] * 3 for _ in range(2)]
dung[0][0] = 9
print("[[0]*3]*2 sau khi gán ô (0,0):", sai, "| tạo bằng vòng for:", dung)

[[0]*3]*2 sau khi gán ô (0,0): [[9, 0, 0], [9, 0, 0]] | tạo bằng vòng for: [[9, 0, 0], [0, 0, 0]]


## 2.4 Áp dụng vào bài này

Định nghĩa "một phép sửa" đến từ **edit distance** (DP hai chuỗi ở trên) — nhưng ở đây chỉ cần biết khoảng cách có bằng 1 không, nên đi bằng **hai con trỏ**: tìm vị trí lệch đầu tiên rồi so phần còn lại theo từng loại phép sửa (thay / chèn).

# 3. Nền tảng

In [12]:
from typing import List
import random, time

s, t = "abcde", "abXde"
print("vị trí lệch đầu tiên:", next((i for i, (a, b) in enumerate(zip(s, t)) if a != b), None))
print("phần sau vị trí 2 có bằng nhau?", s[3:] == t[3:])

vị trí lệch đầu tiên: 2
phần sau vị trí 2 có bằng nhau? True


`zip` dừng ở chuỗi **ngắn hơn** — tiện để tìm vị trí lệch đầu tiên trong phần chung.

## Độ dài nói lên phép sửa nào có thể xảy ra

| `len(t) − len(s)` | Phép sửa duy nhất có thể |
|---|---|
| `0` | thay một ký tự |
| `1` | chèn vào `s` (hay xóa khỏi `t`) |
| `−1` | xóa khỏi `s` |
| `|…| ≥ 2` | không thể — trả `False` ngay |

Đổi chỗ để luôn có `len(s) <= len(t)`, ta chỉ còn hai trường hợp.

# 4. Baseline: edit distance bằng DP

`dp[i][j]` = số phép sửa ít nhất biến `s[:i]` thành `t[:j]`. Đáp án là `dp[n][m] == 1`. `O(nm)` — với `10^4 × 10^4` là quá chậm, nhưng là đáp án đối chiếu tốt cho chuỗi nhỏ.

In [13]:
def edit_distance(s: str, t: str) -> int:
    n, m = len(s), len(t)
    prev = list(range(m + 1))
    for i in range(1, n + 1):
        cur = [i] + [0] * m
        for j in range(1, m + 1):
            if s[i - 1] == t[j - 1]:
                cur[j] = prev[j - 1]
            else:
                cur[j] = 1 + min(prev[j - 1], prev[j], cur[j - 1])   # thay, xóa, chèn
        prev = cur
    return prev[m]


def is_one_edit_dp(s: str, t: str) -> bool:
    return edit_distance(s, t) == 1

for a, b in [("ab", "acb"), ("", ""), ("a", ""), ("abc", "abd"), ("abc", "bcd")]:
    print(f"{a!r:>6} {b!r:>6}: khoảng cách = {edit_distance(a, b)}")

  'ab'  'acb': khoảng cách = 1
    ''     '': khoảng cách = 0
   'a'     '': khoảng cách = 1
 'abc'  'abd': khoảng cách = 1
 'abc'  'bcd': khoảng cách = 2


# 5. Ý tưởng chính

Giả sử `len(s) <= len(t)` và `len(t) − len(s) <= 1`. Tìm `i` = vị trí lệch **đầu tiên** (`s[i] != t[i]`).

- **Cùng độ dài:** phép sửa duy nhất là thay `s[i]` → phần còn lại phải khớp: `s[i+1:] == t[i+1:]`.
- **`t` dài hơn 1:** phép sửa duy nhất là chèn `t[i]` vào trước `s[i]` → `s[i:] == t[i+1:]`.
- **Không có vị trí lệch** trong phần chung: hai chuỗi bằng nhau trên `len(s)` ký tự đầu → đúng khi và chỉ khi `t` dài hơn **đúng 1** (chèn ký tự cuối).

```text
if len(s) > len(t): đổi chỗ
if len(t) - len(s) > 1: return False
for i in range(len(s)):
    if s[i] != t[i]:
        if len(s) == len(t): return s[i+1:] == t[i+1:]
        return s[i:] == t[i+1:]
return len(t) - len(s) == 1
```

In [14]:
def is_one_edit(s: str, t: str) -> bool:
    if len(s) > len(t):
        s, t = t, s
    if len(t) - len(s) > 1:
        return False
    for i in range(len(s)):
        if s[i] != t[i]:
            if len(s) == len(t):
                return s[i + 1:] == t[i + 1:]
            return s[i:] == t[i + 1:]
    return len(t) - len(s) == 1

for a, b in [("ab", "acb"), ("", ""), ("a", ""), ("abc", "abd"), ("abc", "bcd")]:
    print(f"{a!r:>6} {b!r:>6}: {is_one_edit(a, b)}")

  'ab'  'acb': True
    ''     '': False
   'a'     '': True
 'abc'  'abd': True
 'abc'  'bcd': False


# 6. Mô phỏng

In [15]:
def trace_one_edit(s, t):
    print(f"s = {s!r}, t = {t!r}")
    if len(s) > len(t):
        s, t = t, s
        print(f"  đổi chỗ → s = {s!r}, t = {t!r}")
    if len(t) - len(s) > 1:
        print("  độ dài lệch > 1 → False")
        return False
    for i in range(len(s)):
        if s[i] != t[i]:
            if len(s) == len(t):
                r = s[i + 1:] == t[i + 1:]
                print(f"  lệch tại i = {i} ({s[i]!r} ≠ {t[i]!r}), cùng độ dài → so {s[i + 1:]!r} với {t[i + 1:]!r}: {r}")
            else:
                r = s[i:] == t[i + 1:]
                print(f"  lệch tại i = {i} ({s[i]!r} ≠ {t[i]!r}), t dài hơn → so {s[i:]!r} với {t[i + 1:]!r}: {r}")
            return r
    r = len(t) - len(s) == 1
    print(f"  không lệch trong phần chung → {'t dài hơn 1' if r else 'hai chuỗi bằng nhau'}: {r}")
    return r

for a, b in [("abcde", "abXde"), ("acb", "ab"), ("abc", "abc"), ("ab", "abc"), ("abc", "bcd")]:
    trace_one_edit(a, b)

s = 'abcde', t = 'abXde'
  lệch tại i = 2 ('c' ≠ 'X'), cùng độ dài → so 'de' với 'de': True
s = 'acb', t = 'ab'
  đổi chỗ → s = 'ab', t = 'acb'
  lệch tại i = 1 ('b' ≠ 'c'), t dài hơn → so 'b' với 'b': True
s = 'abc', t = 'abc'
  không lệch trong phần chung → hai chuỗi bằng nhau: False
s = 'ab', t = 'abc'
  không lệch trong phần chung → t dài hơn 1: True
s = 'abc', t = 'bcd'
  lệch tại i = 0 ('a' ≠ 'b'), cùng độ dài → so 'bc' với 'cd': False


# 7. Phiên bản đúng format LeetCode

In [16]:
class Solution:
    def isOneEditDistance(self, s: str, t: str) -> bool:
        if len(s) > len(t):
            s, t = t, s
        if len(t) - len(s) > 1:
            return False
        for i in range(len(s)):
            if s[i] != t[i]:
                if len(s) == len(t):
                    return s[i + 1:] == t[i + 1:]
                return s[i:] == t[i + 1:]
        return len(t) - len(s) == 1

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


sol = Solution()
check("Chèn", sol.isOneEditDistance("ab", "acb"), True)
check("Hai rỗng", sol.isOneEditDistance("", ""), False)
check("Xóa", sol.isOneEditDistance("a", ""), True)
check("Thay", sol.isOneEditDistance("abc", "abd"), True)
check("Hai phép", sol.isOneEditDistance("abc", "bcd"), False)
check("Bằng nhau", sol.isOneEditDistance("abc", "abc"), False)
check("Chèn cuối", sol.isOneEditDistance("abc", "abcd"), True)
check("Chữ hoa", sol.isOneEditDistance("a", "A"), True)

random.seed(161)
for _ in range(20000):
    a = "".join(random.choice("ab") for _ in range(random.randint(0, 6)))
    b = "".join(random.choice("ab") for _ in range(random.randint(0, 6)))
    if random.random() < 0.5 and a:            # tạo thêm nhiều cặp gần nhau
        i = random.randrange(len(a))
        b = random.choice([a[:i] + a[i + 1:], a[:i] + "b" + a[i:], a[:i] + "ab"[a[i] == "a"] + a[i + 1:]])
    assert sol.isOneEditDistance(a, b) == is_one_edit_dp(a, b), (a, b)
print("20000 cặp ngẫu nhiên: khớp edit distance == 1")

big = "".join(random.choice("abcXYZ019") for _ in range(10**4))
t0 = time.time()
r1 = sol.isOneEditDistance(big, big[:5000] + "Q" + big[5000:])
r2 = sol.isOneEditDistance(big, big[:-1] + ("a" if big[-1] != "a" else "b"))
t1 = time.time()
print(f"len = 10^4: {r1}, {r2} trong {t1 - t0:.4f}s")

Chèn             PASS  True
Hai rỗng         PASS  False
Xóa              PASS  True
Thay             PASS  True
Hai phép         PASS  False
Bằng nhau        PASS  False
Chèn cuối        PASS  True
Chữ hoa          PASS  True


20000 cặp ngẫu nhiên: khớp edit distance == 1
len = 10^4: True, True trong 0.0007s


# 8. Vì sao đúng? Độ phức tạp?

Gọi `i` là vị trí lệch đầu tiên (`s[:i] == t[:i]`).

- **Cùng độ dài:** chèn/xóa làm đổi độ dài, nên phép sửa duy nhất là thay. Thay ở vị trí `j < i` thì `s[j]` phải khác `t[j]` — mâu thuẫn; thay ở `j > i` thì `s[i]` vẫn khác `t[i]` — sai. Vậy phải thay đúng ở `i`, phần sau phải khớp.
- **`t` dài hơn 1:** phép sửa duy nhất là chèn một ký tự `c` vào `s` tại vị trí `j`. Nếu `j > i`: ký tự thứ `i` của chuỗi mới vẫn là `s[i] ≠ t[i]` — sai. Nếu `j ≤ i`: phần từ vị trí `i + 1` của chuỗi mới chính là `s[i:]`, nên **cần** `s[i:] == t[i+1:]`. Ngược lại, nếu `s[i:] == t[i+1:]` thì chèn `t[i]` tại `i` cho `s[:i] + t[i] + s[i:] = t[:i] + t[i] + t[i+1:] = t` — **đủ**.
- **Không lệch:** `s` là tiền tố của `t`; đúng một phép sửa ⟺ `t` dài hơn đúng 1.

```text
Time:  O(n) — duyệt tìm vị trí lệch + một lần so sánh đuôi
Space: O(n) do cắt chuỗi (slicing tạo bản sao); có thể O(1) bằng hai con trỏ (bài tập 5)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Trả `True` khi hai chuỗi bằng nhau

In [18]:
def wrong_equal(s, t):
    if len(s) > len(t):
        s, t = t, s
    if len(t) - len(s) > 1:
        return False
    for i in range(len(s)):
        if s[i] != t[i]:
            return s[i + 1:] == t[i + 1:] if len(s) == len(t) else s[i:] == t[i + 1:]
    return True                                    # SAI: 0 phép sửa không phải 1

print("Sai :", wrong_equal("abc", "abc"), "| Đúng:", sol.isOneEditDistance("abc", "abc"))

Sai : True | Đúng: False


## Lỗi 2: Quên đổi chỗ để `s` luôn ngắn hơn

In [19]:
def wrong_no_swap(s, t):
    if abs(len(t) - len(s)) > 1:
        return False
    for i in range(min(len(s), len(t))):
        if s[i] != t[i]:
            if len(s) == len(t):
                return s[i + 1:] == t[i + 1:]
            return s[i:] == t[i + 1:]              # chỉ đúng khi t dài hơn
    return abs(len(t) - len(s)) == 1

print("Sai :", wrong_no_swap("acb", "ab"), "| Đúng:", sol.isOneEditDistance("acb", "ab"))

Sai : False | Đúng: True


Với `s = "acb"`, `t = "ab"` (cần **xóa**), công thức `s[i:] == t[i+1:]` so `"cb"` với `""`. Đổi chỗ ở đầu hàm loại bỏ hẳn trường hợp này.

## Lỗi 3: Đếm số vị trí lệch (khoảng cách Hamming) cho mọi trường hợp

In [20]:
def wrong_hamming(s, t):
    if abs(len(s) - len(t)) > 1:
        return False
    diff = sum(a != b for a, b in zip(s, t)) + abs(len(s) - len(t))
    return diff == 1

print("Sai :", wrong_hamming("abc", "xabc"), "| Đúng:", sol.isOneEditDistance("abc", "xabc"))

Sai : False | Đúng: True


Chèn ở **đầu** làm lệch mọi vị trí phía sau. Đếm lệch theo vị trí chỉ đúng với phép **thay** (cùng độ dài).

## Lỗi 4: Dùng DP edit distance cho `10^4 × 10^4`

`10^8` ô — quá chậm trong Python. Bài này chỉ cần biết khoảng cách có **bằng 1** không, nên `O(n)` là đủ.

# 10. Mẹo nhận diện pattern

- **"Cách nhau đúng/tối đa một thao tác"** → tìm vị trí lệch đầu tiên, xử lý theo từng loại thao tác, phần còn lại phải khớp.
- **Biết trước độ dài chênh** → loại bớt trường hợp (đổi chỗ cho `s` ngắn hơn).
- **Khoảng cách tổng quát** → DP edit distance (#72); **khoảng cách ≤ k** → DP trong dải đường chéo rộng `2k + 1`.

Bài liên quan: **#72 Edit Distance**, **#583 Delete Operation for Two Strings**, **#1790 Check if One String Swap Can Make Strings Equal** (Easy), **#859 Buddy Strings** (Easy), **#680 Valid Palindrome II** (Easy, cùng ý "bỏ một ký tự khi lệch").

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Áp dụng thuật toán cho: `("teacher", "teachar")`, `("teacher", "treacher")`, `("teacher", "teach")`, `("", "x")`.

## Bài 2 — Điền chỗ trống

In [21]:
def exercise_2_is_one_edit(s: str, t: str) -> bool:
    if len(s) > len(t):
        s, t = t, s
    if len(t) - len(s) > 1:
        return False
    for i in range(len(s)):
        # TODO: khi s[i] != t[i], so phần còn lại theo hai trường hợp
        pass
    return len(t) - len(s) == 1


# assert exercise_2_is_one_edit("ab", "acb") is True
# assert exercise_2_is_one_edit("", "") is False
# assert exercise_2_is_one_edit("abc", "abd") is True
# assert exercise_2_is_one_edit("abc", "bcd") is False

## Bài 3 — Biến thể: **tối đa** một phép sửa (0 hoặc 1)

(Câu hỏi phỏng vấn "One Away" trong *Cracking the Coding Interview*.)

In [22]:
def exercise_3_at_most_one_edit(s: str, t: str) -> bool:
    # TODO
    pass


# assert exercise_3_at_most_one_edit("pale", "ple") is True
# assert exercise_3_at_most_one_edit("pale", "pale") is True
# assert exercise_3_at_most_one_edit("pale", "bake") is False

## Bài 4 — Biến thể: Edit Distance (LeetCode #72)

Số phép sửa **ít nhất** để biến `word1` thành `word2`. Viết DP hai hàng, `O(nm)` thời gian, `O(m)` bộ nhớ.

In [23]:
def exercise_4_min_distance(word1: str, word2: str) -> int:
    # TODO
    pass


# assert exercise_4_min_distance("horse", "ros") == 3
# assert exercise_4_min_distance("intention", "execution") == 5
# assert exercise_4_min_distance("", "abc") == 3

## Bài 5 — Mini challenge: `O(1)` bộ nhớ phụ

Viết lại lời giải **không cắt chuỗi** (slicing tạo bản sao `O(n)`): dùng hai con trỏ `i`, `j` cùng đi; khi gặp lệch lần đầu thì "tiêu" phép sửa bằng cách chỉ tăng `j` (chèn) hoặc tăng cả hai (thay); lệch lần hai → `False`.

In [24]:
def exercise_5_is_one_edit_o1(s: str, t: str) -> bool:
    # TODO
    pass


# assert exercise_5_is_one_edit_o1("ab", "acb") is True
# assert exercise_5_is_one_edit_o1("acb", "ab") is True
# assert exercise_5_is_one_edit_o1("abc", "abc") is False
# assert exercise_5_is_one_edit_o1("abc", "bcd") is False
# assert exercise_5_is_one_edit_o1("", "a") is True

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_one_edit` sau khi tự làm.

**Bài 3**
- Mức 1: `s == t or is_one_edit(s, t)`; hoặc sửa dòng cuối thành `return True` (vì độ dài lệch ≤ 1 đã được kiểm tra).

**Bài 4**
- Mức 1: `dp[i][j] = dp[i-1][j-1]` nếu ký tự bằng, ngược lại `1 + min(thay, xóa, chèn)`.
- Mức 2: hàng đầu `0..m`; mỗi hàng mới bắt đầu bằng `i`.

**Bài 5**
- Mức 1: đổi chỗ cho `len(s) <= len(t)`; `edited = False`.
- Mức 2: khi `s[i] != t[j]`: nếu `edited` → `False`; `edited = True`; nếu cùng độ dài thì `i += 1`; luôn `j += 1`. Khi bằng: tăng cả hai.
- Mức 3: kết thúc: nếu chưa sửa thì cần `len(t) - len(s) == 1`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [25]:
for a, b in [("teacher", "teachar"), ("teacher", "treacher"), ("teacher", "teach"), ("", "x")]:
    print("→", trace_one_edit(a, b))

s = 'teacher', t = 'teachar'
  lệch tại i = 5 ('e' ≠ 'a'), cùng độ dài → so 'r' với 'r': True
→ True
s = 'teacher', t = 'treacher'
  lệch tại i = 1 ('e' ≠ 'r'), t dài hơn → so 'eacher' với 'eacher': True
→ True
s = 'teacher', t = 'teach'
  đổi chỗ → s = 'teach', t = 'teacher'
  độ dài lệch > 1 → False
→ False
s = '', t = 'x'
  không lệch trong phần chung → t dài hơn 1: True
→ True


## Đáp án Bài 2

In [26]:
def answer_2_is_one_edit(s: str, t: str) -> bool:
    return Solution().isOneEditDistance(s, t)

assert answer_2_is_one_edit("ab", "acb") is True
assert answer_2_is_one_edit("", "") is False
assert answer_2_is_one_edit("abc", "abd") is True
assert answer_2_is_one_edit("abc", "bcd") is False
print("Bài 2 PASS (giống Solution ở mục 7)")

Bài 2 PASS (giống Solution ở mục 7)


## Đáp án Bài 3

In [27]:
def answer_3_at_most_one_edit(s: str, t: str) -> bool:
    return s == t or Solution().isOneEditDistance(s, t)

assert answer_3_at_most_one_edit("pale", "ple") is True
assert answer_3_at_most_one_edit("pale", "pale") is True
assert answer_3_at_most_one_edit("pale", "bake") is False
for _ in range(5000):
    a = "".join(random.choice("ab") for _ in range(random.randint(0, 5)))
    b = "".join(random.choice("ab") for _ in range(random.randint(0, 5)))
    assert answer_3_at_most_one_edit(a, b) == (edit_distance(a, b) <= 1)
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [28]:
def answer_4_min_distance(word1: str, word2: str) -> int:
    return edit_distance(word1, word2)

assert answer_4_min_distance("horse", "ros") == 3
assert answer_4_min_distance("intention", "execution") == 5
assert answer_4_min_distance("", "abc") == 3
print("Bài 4 PASS (chính là edit_distance ở mục 4)")

Bài 4 PASS (chính là edit_distance ở mục 4)


## Đáp án Bài 5

In [29]:
def answer_5_is_one_edit_o1(s: str, t: str) -> bool:
    if len(s) > len(t):
        s, t = t, s
    n, m = len(s), len(t)
    if m - n > 1:
        return False
    i = j = 0
    edited = False
    while i < n and j < m:
        if s[i] == t[j]:
            i += 1; j += 1
        else:
            if edited:
                return False
            edited = True
            if n == m:
                i += 1                       # thay
            j += 1                           # chèn (hoặc phần j của phép thay)
    if edited:
        return True                          # nếu t dài hơn, ký tự thừa đã được "tiêu" ở lần lệch
    return m - n == 1

assert answer_5_is_one_edit_o1("ab", "acb") is True
assert answer_5_is_one_edit_o1("acb", "ab") is True
assert answer_5_is_one_edit_o1("abc", "abc") is False
assert answer_5_is_one_edit_o1("abc", "bcd") is False
assert answer_5_is_one_edit_o1("", "a") is True
for _ in range(20000):
    a = "".join(random.choice("ab") for _ in range(random.randint(0, 6)))
    b = "".join(random.choice("ab") for _ in range(random.randint(0, 6)))
    assert answer_5_is_one_edit_o1(a, b) == is_one_edit_dp(a, b), (a, b)
print("Bài 5 PASS")

Bài 5 PASS


Vì sao `if edited: return True` không bỏ sót ký tự thừa? Khi `m = n + 1` và đã lệch một lần, `j` đã tiến hơn `i` đúng 1; vòng lặp chỉ dừng không lỗi khi `i == n`, lúc đó `j == m`. Khi `m == n`, `i` và `j` luôn bằng nhau nên cũng kết thúc cùng lúc.

# 14. Tổng kết

> Đổi chỗ cho `s` ngắn hơn; lệch độ dài > 1 → `False`. Tìm vị trí lệch đầu tiên `i`: cùng độ dài → `s[i+1:] == t[i+1:]`; `t` dài hơn → `s[i:] == t[i+1:]`. Không lệch → đúng khi `t` dài hơn **đúng 1** (hai chuỗi bằng nhau là `False`). `O(n)`.

---

## Checklist trước khi nộp

- [ ] Đổi chỗ để `len(s) <= len(t)`?
- [ ] Lệch độ dài > 1 → `False` ngay?
- [ ] Hai chuỗi bằng nhau → `False`?
- [ ] Chèn ở cuối (`"abc"`, `"abcd"`) → `True`?